Getting started with binary analysis
====================================

`ReProspect` spans three complementary forms of analysis of CUDA code: API tracing, kernel profiling, and binary analysis.
Here, we show how to get started with binary analysis.

Binary analysis examines compiled CUDA code.
The CUDA toolkit provides the CUDA binary utilities `cuobjdump` and `nvdisasm` for extracting and disassembling the contents of CUDA binary files.
`ReProspect` enables a fully programmatic use of these tools: it launches them, collects their output into `Python` data structures, and supports analyses of these data that can go up to test assertions.

As an example, we consider a kernel involving an atomic function call.
We examine the CUDA assembly (SASS) instructions that this call is compiled to.

About this page
---------------

This page is rendered from a Jupyter notebook, executed at documentation build time, located at `docs/source/getting-started/example_binary_analysis.ipynb` in the repository.

To run the example yourself, {download}`download the notebook <example_binary_analysis.ipynb>` and open it in JupyterLab.
Alternatively, copy-paste the code snippets into an interactive Python session.

The requirements are Python 3.10 or newer, `ReProspect` and JupyterLab,

```bash
python -m pip install reprospect jupyterlab
```

and a CUDA Toolkit 12.8 or newer installation providing `nvcc`, `cuobjdump`, and `cu++filt`.
Because the example is concerned with analysing the assembly code produced by compilation but does not run the kernel, no GPU is required.

Source code
-----------

Let us consider a kernel that atomically adds each element of a source array to the corresponding element of a destination array.

In this source code, `atomicAdd` is an atomic function provided by CUDA as a `C++` language extension.
It reads the value from a given memory address, adds a value to it, and writes the result back to the same memory address, all in one atomic operation, as described [here](https://docs.nvidia.com/cuda/cuda-c-programming-guide/#atomicadd).
It should be noted that `atomicAdd` also returns the value that was previously stored at the address, but this kernel does not use this return value.

In [ ]:
CODE = """\
#include "cuda.h"

__global__ void my_kernel(int* __restrict__ const dst, const int* __restrict__ const src) {
    const auto index = blockIdx.x * blockDim.x + threadIdx.x;
    atomicAdd(&dst[index], src[index]);
}
"""

Compilation
-----------

We compile the source code for three targets spanning the Ampere architecture (compute capability 8.0, target `sm_80`), Hopper (9.0, `sm_90`), and Blackwell (12.0, `sm_120`).

Later in this notebook, the atomic function call will be found to compile to different SASS instructions across these targets.

In [ ]:
import pathlib
import subprocess

from reprospect.tools import architecture

ARCHES = [
    architecture.NVIDIAArch.from_compute_capability(80),
    architecture.NVIDIAArch.from_compute_capability(90),
    architecture.NVIDIAArch.from_compute_capability(120),
]

print(subprocess.check_output(('nvcc', '--version')).decode())

workdir = pathlib.Path.cwd() / 'example_binary_analysis'
workdir.mkdir(exist_ok=True)

for arch in ARCHES:
    source = workdir / f'atomic.{arch.as_sm}.cu'
    output = workdir / f'atomic.{arch.as_sm}'

    source.write_text(CODE)

    subprocess.check_call(('nvcc', f'--generate-code=arch={arch.as_compute},code=[{arch.as_sm}]', '-O3', '-c', source, '-o', output))

Extracting and disassembling CUDA binary code
---------------------------------------------

We now use `ReProspect` to extract and disassemble the CUDA binary code from the compiled output file for each target.

The CUDA toolkit provides `cuobjdump` for this purpose.
Here, we invoke it through the `ReProspect` class {py:class}`reprospect.tools.binaries.cuobjdump.CuObjDump`.
The method {py:meth}`~reprospect.tools.binaries.cuobjdump.CuObjDump.extract` is a factory method: it extracts, from the compiled output file, the embedded CUDA binary file (`CUBIN`) for the target architecture, and it disassembles the SASS instructions of each kernel in this `CUBIN`.
The returned object has a member `functions`, a dictionary that maps each kernel's demangled signature to its SASS code as a string; we will use this demangled signature below to select our kernel.

It should be noted that the CUDA binary files embedded in a compiled output file follow a naming convention as in `<basename>.<index>.<sm>.cubin`.
The `ReProspect` class {py:class}`~reprospect.tools.binaries.cuobjdump.CuObjDump` has a method {py:meth}`~reprospect.tools.binaries.cuobjdump.CuObjDump.embedded_cubins` to inspect the names of the CUDA binary files embedded in a compiled output file.

In [ ]:
from reprospect.tools.binaries import CuObjDump

cuobjdump = {}

for arch in ARCHES:
    compiled_output = workdir / f'atomic.{arch.as_sm}'

    cuobjdump[arch], _ = CuObjDump.extract(
        file=compiled_output,
        arch=arch,
        cwd=workdir,
        cubin=f'atomic.1.{arch.as_sm}.cubin',
    )

    print(cuobjdump[arch].functions.keys())

Like many classes in `ReProspect`, the class {py:class}`reprospect.tools.binaries.cuobjdump.CuObjDump` supports rich rendering.
Printing shows for each kernel its demangled signature, its SASS code, and information about its resource usage, such as the number of general purpose registers that the kernel uses.

In [ ]:
for arch in ARCHES:
    print(cuobjdump[arch])

Decoding SASS instructions
--------------------------

Next, we use `ReProspect` to parse the SASS instructions.
`ReProspect` provides the class {py:class}`reprospect.tools.binaries.sass.decoder.Decoder` for this purpose.
For each target, an instance of this class is constructed from the SASS code stored as a string.
This instance has a member `instructions`, a list wherein each SASS instruction is represented by its offset, its hexadecimal encoding, its disassembled representation as a string, and its decoded control code.
The further decomposition of the disassembled representation as a string into its components (predicate, opcode, modifiers, and operands) is performed by the instruction matchers introduced below.

In [ ]:
from reprospect.tools.binaries.sass import Decoder

SIGNATURE = 'my_kernel(int *, const int *)'

decoder: dict[architecture.NVIDIAArch, Decoder] = {}

for arch in ARCHES:
    decoder[arch] = Decoder(code=cuobjdump[arch].functions[SIGNATURE].code)

The class {py:class}`~reprospect.tools.binaries.sass.decoder.Decoder` supports rich rendering.
Printing shows for each SASS instruction its offset, its disassembled representation as a string, and its decoded control code.

In [ ]:
for arch in ARCHES:
    print(decoder[arch])

We can observe that the kernel defined above compiles to a sequence of SASS instructions, including load/store instructions (such as those with opcode `LDCU`, `LDC`, and `LDG` in the case of the `sm_120` target), and integer instructions (such as those with opcode `IMAD` for `sm_120`, used here for address computation).

With the displayed CUDA Toolkit version, `-O3`, and the selected targets, this kernel’s unused-result `atomicAdd` lowers to one *Reduction Operation on Generic Memory* SASS instruction: `RED` on `sm_80` and `REDG` on the newer architecture targets.

We can assert this behavior by checking that the decoded instruction list contains exactly one instruction whose representation has the expected opcode. 
It should be noted that when proceeding in this way, the opcode must be hard-coded per target architecture.
The matchers introduced next will abstract such details away.

In [ ]:
instruction_red = {}

for arch in ARCHES:
    opcode = 'RED' if arch.compute_capability < 90 else 'REDG'
    instruction_red[arch] = [
        instr for instr in decoder[arch].instructions
        if f'{opcode}.' in instr.instruction
    ]
    assert len(instruction_red[arch]) == 1

Decomposing SASS instructions
-----------------------------

`ReProspect` provides an extensible matching framework for SASS instructions.
At the lowest levels, matchers analyse instructions and their components (predicate, opcode, modifiers, and operands).

In [ ]:
from reprospect.testing.binaries.sass.instruction import AnyMatcher

for arch in ARCHES:
    decomposed = AnyMatcher().match(inst=instruction_red[arch][0].instruction)
    print(f'Arch: {arch.as_sm}, Instruction: {decomposed}')

We can observe that for each considered target, the modifiers are `E`, `ADD`, `STRONG` and `GPU`.
These modifiers indicate extended addressing (`E`), the operation (`ADD`), the consistency (`STRONG`), and the scope (`GPU`, i.e., `Device`).
NVIDIA documents SASS instructions only to a very limited extent; their interpretation often stems from community reverse-engineering efforts.
Further, although the modifiers coincide across the three targets here, modifiers may depend on the target architecture in general.

`ReProspect`'s matchers are implemented internally in terms of regex matching.

In [ ]:
print(AnyMatcher().PATTERN)

Matching SASS instructions across target architectures
------------------------------------------------------

`ReProspect` provides instruction matchers that abstract away target-architecture-specific details.

Here, we use {py:class}`reprospect.testing.binaries.sass.instruction.atomic.ReductionMatcher`, whose internal implementation encodes how the instruction components of a *Reduction Operation on Generic Memory* instruction depend on the target architecture.
It builds its regex pattern as a function of the target architecture and of the requested operation, consistency, scope, and value type.

In [ ]:
import numpy

from reprospect.testing.binaries.sass.instruction import ReductionMatcher, ThreadScope

for arch in ARCHES:
    matcher_red = ReductionMatcher(
        arch=arch, operation='ADD',
        scope=ThreadScope.DEVICE,
        consistency='STRONG', dtype=numpy.int32,
    )
    print(f'Arch: {arch.as_sm}, Pattern: {matcher_red.pattern}')
    matched_red = [
        (inst, matched)
        for inst in decoder[arch].instructions
        if (matched := matcher_red.match(inst))
    ]
    assert len(matched_red) == 1
    print(f'Arch: {arch.as_sm}, Instruction: {matched_red}')

The parameters passed to {py:class}`~reprospect.testing.binaries.sass.instruction.atomic.ReductionMatcher` mirror the source code: `atomicAdd` on `int` acts at device scope with strong consistency.

Thus, a single matcher asserts the presence of the reduction instruction across all targets: the architecture-specific details are encapsulated in the matcher, and no opcode or modifier is hard-coded.

Outlook
-------

This notebook illustrated how `ReProspect` supports a fully programmatic analysis of compiled CUDA code.
Matching SASS instructions has practical value.
For example, staying in the context of atomic functions, a code base may provide several implementations of an atomic operation, such as a mapping to a hardware atomic instruction, a compare-and-swap loop, or a lock-based implementation, with substantially different performance.
Which implementation is selected may depend on intricate logic in the code base and the compiler tool chain, and may change as either evolves.
Verifying the selection by micro-benchmarking requires a physical device and suffers from runtime variability; the compiled CUDA code, by contrast, already contains the answer.
With SASS instruction matchers, the intended mapping can become a test assertion that can run in CI/CD pipelines.
With `ReProspect`'s target-architecture-aware SASS instruction matchers, such a test assertion can be written in terms of a single matcher.
Such a use case is exemplified in our `Kokkos` atomics with desul example.